In [ ]:
WDN_NAME = ""
BASE_DIR = f"old/data/{WDN_NAME}"
IGNORE_MEASUREMENTS = True

# Evaluation
Generate R² scatter and AED visualizations for the test split.

In [ ]:
import json
import os
from pathlib import Path
import importlib.util
import pickle

import numpy as np
import torch
import networkx as nx
from torch_geometric.nn import GCNConv

In [ ]:
metrics_path = Path("algorithms/8_evaluate-model/metrics.py")
metrics_spec = importlib.util.spec_from_file_location("metrics_module", metrics_path)
metrics_module = importlib.util.module_from_spec(metrics_spec)
metrics_spec.loader.exec_module(metrics_module)
calculate_node_metrics = metrics_module.calculate_node_metrics
calculate_global_metrics = metrics_module.calculate_global_metrics
create_scatter_plot = metrics_module.create_scatter_plot

import algorithms.registry as registry

def _noop_register(_name):
    def decorator(func):
        return func
    return decorator

registry.REGISTRY.register = _noop_register

viz_path = Path("algorithms/9_visualize-evaluation/visualize_evaluation.py")
viz_spec = importlib.util.spec_from_file_location("viz_module", viz_path)
viz_module = importlib.util.module_from_spec(viz_spec)
viz_spec.loader.exec_module(viz_module)
create_advanced_error_distribution_viz = viz_module.create_advanced_error_distribution_viz

In [ ]:
base_dir = Path(BASE_DIR)
data_dir = base_dir / "data_generator"
gnn_dir = base_dir / "gnn_model"
eval_dir = base_dir / "evaluation"
eval_dir.mkdir(parents=True, exist_ok=True)

graph_path = data_dir / "graph_with_measurements.pickle"
artifacts_path = data_dir / "evaluation_artifacts.json"
stats_path = data_dir / "dataset_stats.json"
metadata_path = data_dir / "dataset_metadata.json"

with graph_path.open("rb") as f:
    G = pickle.load(f)
with artifacts_path.open("r", encoding="utf-8") as f:
    artifacts = json.load(f)

dataset_stats = {}
if stats_path.exists():
    with stats_path.open("r", encoding="utf-8") as f:
        dataset_stats = json.load(f)
dataset_metadata = {}
if metadata_path.exists():
    with metadata_path.open("r", encoding="utf-8") as f:
        dataset_metadata = json.load(f)

In [ ]:
wdn_config_path = Path("wdn") / f"{WDN_NAME}.json"
wdn_config = {}
if wdn_config_path.exists():
    with wdn_config_path.open("r", encoding="utf-8") as f:
        wdn_config = json.load(f)

aed_scale = wdn_config.get("scale", 1.0)
aed_node_scale = wdn_config.get("node_scale", 1.0)
aed_font_scale = wdn_config.get("font_scale", 1.0)
aed_node_label_threshold = wdn_config.get("node_label_threshold", 0.01)
aed_non_special_node_scale = wdn_config.get("non_special_node_scale", 1.0)

In [ ]:
train_data = torch.load(data_dir / "train_dataset.pt", weights_only=False)
val_data = torch.load(data_dir / "val_dataset.pt", weights_only=False)
test_data = torch.load(data_dir / "test_dataset.pt", weights_only=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class GCN(torch.nn.Module):
    def __init__(self, dim_in, dim_h, dim_out):
        super().__init__()
        self.dim_hidden = dim_h * 4
        self.batch_norm1 = torch.nn.BatchNorm1d(self.dim_hidden)
        self.batch_norm2 = torch.nn.BatchNorm1d(self.dim_hidden)
        self.batch_norm3 = torch.nn.BatchNorm1d(self.dim_hidden)
        self.gcn1 = GCNConv(dim_in, self.dim_hidden, improved=False, cached=False)
        self.gcn2 = GCNConv(self.dim_hidden, self.dim_hidden, improved=False, cached=False)
        self.gcn3 = GCNConv(self.dim_hidden, self.dim_hidden, improved=False, cached=False)
        self.linear1 = torch.nn.Linear(self.dim_hidden, dim_h)
        self.linear2 = torch.nn.Linear(dim_h, dim_out)
        self.dropout = torch.nn.Dropout(p=0.2)

    def forward(self, x, edge_index, edge_attr=None):
        h = self.gcn1(x, edge_index, edge_attr)
        h = self.batch_norm1(h)
        h = torch.relu(h)
        h = self.dropout(h)
        h2 = self.gcn2(h, edge_index, edge_attr)
        h2 = self.batch_norm2(h2)
        h2 = torch.relu(h2)
        h2 = self.dropout(h2)
        h2 = h2 + h
        h3 = self.gcn3(h2, edge_index, edge_attr)
        h3 = self.batch_norm3(h3)
        h3 = torch.relu(h3)
        h3 = self.dropout(h3)
        h3 = h3 + h2
        h = self.linear1(h3)
        h = torch.relu(h)
        h = self.dropout(h)
        h = self.linear2(h)
        return h

input_dim = train_data[0].x.shape[1]
model = GCN(dim_in=input_dim, dim_h=256, dim_out=1).to(device)
model_path = gnn_dir / "best_model.pt"
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

In [ ]:
def run_inference(model, dataset, device):
    predictions = []
    actuals = []
    masks = []
    with torch.no_grad():
        for data in dataset:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr)
            predictions.append(out.squeeze().cpu().numpy())
            actuals.append(data.y.squeeze().cpu().numpy())
            masks.append(data.mask.cpu().numpy())
    return np.stack(predictions), np.stack(actuals), np.stack(masks)

predictions, actuals, node_masks = run_inference(model, test_data, device)

In [ ]:
def _indices_from_mapping(node_mapping, target_nodes):
    if not node_mapping or not target_nodes:
        return []
    target_strs = {str(n) for n in target_nodes}
    indices = []
    for idx_str, node in node_mapping.items():
        if str(node) in target_strs:
            try:
                indices.append(int(idx_str))
            except ValueError:
                continue
    return indices

def _indices_from_list(node_list, target_nodes):
    if not node_list or not target_nodes:
        return []
    target_strs = {str(n) for n in target_nodes}
    return [idx for idx, node in enumerate(node_list) if str(node) in target_strs]

node_mapping = artifacts.get("node_mapping", {})
node_list = artifacts.get("node_list", list(G.nodes()))
reservoir_nodes = artifacts.get("reservoir_nodes", [])
measurement_nodes = artifacts.get("measurement_nodes", [])
secondary_pipes = artifacts.get("secondary_pipes", [])

virtual_measurement_nodes = [n for n in G.nodes() if str(n).startswith("meas_")]
measurement_nodes_clean = [n for n in measurement_nodes if not str(n).startswith("meas_")]

reservoir_indices = _indices_from_mapping(node_mapping, reservoir_nodes)
measurement_indices = _indices_from_mapping(node_mapping, measurement_nodes_clean)
if not reservoir_indices:
    reservoir_indices = _indices_from_list(node_list, reservoir_nodes)
if not measurement_indices:
    measurement_indices = _indices_from_list(node_list, measurement_nodes_clean)

exclude_r2 = list(dict.fromkeys(reservoir_indices + (measurement_indices if IGNORE_MEASUREMENTS else [])))
exclude_aed = list(dict.fromkeys(reservoir_indices))

node_mask = node_masks[0]

pos = nx.get_node_attributes(G, "pos")
if pos:
    pos_2d = {}
    for node, coords in pos.items():
        if coords is None:
            continue
        if len(coords) >= 2:
            pos_2d[node] = (coords[0], coords[1])
    if pos_2d:
        nx.set_node_attributes(G, pos_2d, "pos")

pressure_range = dataset_stats.get("pressure_range") or dataset_metadata.get("pressure_range")
pred_scatter = predictions
actual_scatter = actuals
if pressure_range and "min" in pressure_range and "max" in pressure_range:
    pr_min = float(pressure_range["min"])
    pr_max = float(pressure_range["max"])
    pred_scatter = predictions * (pr_max - pr_min) + pr_min
    actual_scatter = actuals * (pr_max - pr_min) + pr_min

r2_path = eval_dir / "r2_scatter.png"
create_scatter_plot(pred_scatter, actual_scatter, node_mask, exclude_r2, str(r2_path))
print(f"Saved R^2 scatter to: {r2_path}")

G_aed = G.copy()
if virtual_measurement_nodes:
    G_aed.remove_nodes_from(virtual_measurement_nodes)
if isinstance(G_aed, nx.MultiDiGraph):
    G_aed = nx.DiGraph(G_aed)
elif isinstance(G_aed, nx.MultiGraph):
    G_aed = nx.Graph(G_aed)

node_metrics = calculate_node_metrics(predictions, actuals, node_mask, exclude_aed)
aed_img = create_advanced_error_distribution_viz(
    G_aed,
    node_metrics,
    secondary_pipes,
    measurement_nodes_clean,
    scale=aed_scale,
    node_scale=aed_node_scale,
    font_scale=aed_font_scale,
    node_label_threshold=aed_node_label_threshold,
    non_special_node_scale=aed_non_special_node_scale,
    node_mapping=node_mapping,
 )
aed_path = eval_dir / "aed_viz.png"
if aed_img is not None:
    aed_img.save(aed_path)
    print(f"Saved AED visualization to: {aed_path}")
else:
    print("AED visualization failed")